In [ ]:
# =====================================
# Shared Configuration — run this cell first
# =====================================
import os
from pathlib import Path
import logging
import psycopg2
from sentence_transformers import SentenceTransformer

# ── Database ──────────────────────────────────────────────────────────────────
DB_CONFIG = {
    "dbname":   os.environ.get("PGDATABASE", "agents_db"),
    "user":     os.environ.get("PGUSER",     "gauraangmalik"),
    "password": os.environ.get("PGPASSWORD", ""),        # set PGPASSWORD env var
    "host":     os.environ.get("PGHOST",     "localhost"),
    "port":     int(os.environ.get("PGPORT", "5432")),
}
DB_URI = (
    f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
)

# ── Models ────────────────────────────────────────────────────────────────────
MODELS = ["tinyllama", "qwen3.5:4b", "gemma4:e4b"]  # run sequentially; agents parallel within each

# ── Ollama ────────────────────────────────────────────────────────────────────
OLLAMA_BASE_URL  = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
_timeout_env     = os.environ.get("REQUEST_TIMEOUT")
REQUEST_TIMEOUT  = int(_timeout_env) if _timeout_env else 240   # 240 s = 4-min hard cap per Ollama call
MAX_WORKERS      = int(os.environ.get("MAX_WORKERS",     "5"))
MAX_RESPONSE_LEN = 350                                             # same for both models

# ── System prompts per query type ─────────────────────────────────────────────
SYSTEM_PROMPTS = {
    "factual": (
        "Provide only the most relevant factual response in 3-4 sentences "
        f"(max {MAX_RESPONSE_LEN} characters). "
        "Do NOT include introductions, disclaimers, or AI statements. State the facts."
    ),
    "planning": (
        "Break the task into clear, numbered, actionable steps. "
        f"Be specific and sequential. Max {MAX_RESPONSE_LEN} characters. "
        "No preamble — go straight to step 1."
    ),
    "reasoning": (
        "Think through the problem carefully. Show your reasoning and "
        f"arrive at a well-supported conclusion. Max {MAX_RESPONSE_LEN} characters."
    ),
    "creative": (
        "Generate a creative, imaginative, and detailed response. "
        f"Max {MAX_RESPONSE_LEN} characters. No disclaimers."
    ),
    "default": (
        f"Respond concisely and accurately in 3-4 sentences. Max {MAX_RESPONSE_LEN} characters."
    ),
}

# ── Output directory ─────────────────────────────────────────────────────────
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# ── Logging & cursor ──────────────────────────────────────────────────────────
logging.basicConfig(
    filename=str(RESULTS_DIR / "executed_queries.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

class LoggingCursor(psycopg2.extensions.cursor):
    def execute(self, sql, args=None):
        logging.info(self.mogrify(sql, args).decode("utf-8"))
        try:
            super().execute(sql, args)
        except Exception as e:
            logging.error(f"Error executing query: {e}")
            raise

# ── Embedding model (loaded once; all cells reuse) ────────────────────────────
embed_model = SentenceTransformer("all-MiniLM-L6-v2")


print(f"✅ Config loaded. Models: {MODELS}")
print(f"   Ollama: {OLLAMA_BASE_URL}  timeout={REQUEST_TIMEOUT}s  workers={MAX_WORKERS}")


In [ ]:
# BLOCK A: Multi-Agent Planning, Response Storage, and Semantic Retrieval System
# Requires: shared config cell to be run first
import psycopg2
import pandas as pd
import numpy as np
import requests
import faiss
import time
import re
import pickle
import os
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Optional
from sqlalchemy import create_engine

# =====================================
# Active system prompt (set per-task by process_task)
# =====================================
_ACTIVE_SYSTEM_PROMPT = SYSTEM_PROMPTS["default"]

def get_system_prompt(query_type: str) -> str:
    return SYSTEM_PROMPTS.get(query_type, SYSTEM_PROMPTS["default"])

def clean_response(text: str) -> str:
    remove_phrases = [
        "As an AI,", "I'm not capable of", "I cannot provide personal opinions",
        "Based on available information", "I can provide a factual response",
        "System: The given input", "The given text", "I believe", "I think",
        "It is possible that", "I've come to the conclusion that",
    ]
    for phrase in remove_phrases:
        text = re.sub(rf"(^|\b){re.escape(phrase)}", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"^[,.\s]+", "", text)
    if text and text[0].islower():
        text = text.capitalize()
    return text

# =====================================
# Planning taxonomy
# =====================================
PLANNING_AGENTS = {
    "Planning Without Feedback": {
        "Single-Path Reasoning": ["Chain of Thought (CoT)", "Zero-Shot CoT", "Re-Prompting"],
        "Multi-Path Reasoning":  ["ReWOO", "HuggingGPT", "Tree-of-Thought (ToT)"],
        "External Planner":      ["LLM-Planner", "ReAct", "LLM+P"],
    },
    "Planning With Feedback": {
        "Environment Feedback": ["Inner Monologue", "LLM4RL"],
        "Human Feedback":       ["ChatCoT", "TPTU"],
        "Model Feedback":       ["Self-Refine", "SelfCheck"],
    },
}

def populate_planning_taxonomy():
    with psycopg2.connect(**DB_CONFIG) as conn:
        with conn.cursor() as cur:
            for cat in PLANNING_AGENTS:
                cur.execute("INSERT INTO planning_categories (name) VALUES (%s) ON CONFLICT (name) DO NOTHING;", (cat,))
            conn.commit()
            for cat, subcats in PLANNING_AGENTS.items():
                cur.execute("SELECT id FROM planning_categories WHERE name = %s;", (cat,))
                cat_id = cur.fetchone()[0]
                for sub in subcats:
                    cur.execute("INSERT INTO planning_subcategories (name, category_id) VALUES (%s, %s) ON CONFLICT (name) DO NOTHING;", (sub, cat_id))
            conn.commit()
    print("✅ Planning taxonomy populated.")

# =====================================
# Database setup
# =====================================
def drop_existing_table():
    with psycopg2.connect(**DB_CONFIG, cursor_factory=LoggingCursor) as conn:
        with conn.cursor() as cur:
            for t in ["agent_pair_performance", "task_performance", "agent_responses",
                      "agents", "tasks", "planning_subcategories", "planning_categories"]:
                cur.execute(f"DROP TABLE IF EXISTS {t} CASCADE;")
            conn.commit()
    print("✅ Dropped all tables.")

def setup_database():
    with psycopg2.connect(**DB_CONFIG, cursor_factory=LoggingCursor) as conn:
        with conn.cursor() as cur:
            cur.execute("""
                CREATE TABLE IF NOT EXISTS planning_categories (
                    id SERIAL PRIMARY KEY, name TEXT UNIQUE NOT NULL);""")
            cur.execute("""
                CREATE TABLE IF NOT EXISTS planning_subcategories (
                    id SERIAL PRIMARY KEY, name TEXT UNIQUE NOT NULL,
                    category_id INT REFERENCES planning_categories(id) ON DELETE CASCADE);""")
            cur.execute("""
                CREATE TABLE IF NOT EXISTS agents (
                    id SERIAL PRIMARY KEY,
                    title TEXT UNIQUE NOT NULL,
                    profile TEXT, memory TEXT,
                    planning_category_id INT REFERENCES planning_categories(id) ON DELETE CASCADE,
                    planning_subcategory_id INT REFERENCES planning_subcategories(id) ON DELETE CASCADE,
                    action TEXT, capability_acquisition TEXT,
                    social_science TEXT, natural_science TEXT, engineering TEXT,
                    subjective TEXT, objective TEXT, benchmark TEXT,
                    publication TEXT, code_url TEXT, paper_url TEXT
                );""")
            # NEW: tasks table — stores the query text and metadata per task
            cur.execute("""
                CREATE TABLE IF NOT EXISTS tasks (
                    task_name  TEXT PRIMARY KEY,
                    query_text TEXT NOT NULL,
                    query_type TEXT NOT NULL DEFAULT 'default',
                    domain     TEXT,
                    difficulty TEXT
                );""")
            # NEW: agent_responses — one row per (agent, task, model); replaces TEXT[] array
            cur.execute("""
                CREATE TABLE IF NOT EXISTS agent_responses (
                    id          SERIAL PRIMARY KEY,
                    agent_title TEXT NOT NULL REFERENCES agents(title) ON DELETE CASCADE,
                    task_name   TEXT NOT NULL REFERENCES tasks(task_name) ON DELETE CASCADE,
                    model       TEXT NOT NULL,
                    response    TEXT NOT NULL,
                    timestamp   TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    UNIQUE (agent_title, task_name, model)
                );""")
            # model column added; UNIQUE now includes model
            cur.execute("""
                CREATE TABLE IF NOT EXISTS task_performance (
                    id                     SERIAL PRIMARY KEY,
                    task_name              TEXT NOT NULL,
                    model                  TEXT NOT NULL,
                    avg_euclidean_distance FLOAT,
                    avg_cosine_similarity  FLOAT,
                    completion_time        FLOAT,
                    response_length        INT,
                    timestamp              TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    UNIQUE (task_name, model)
                );""")
            cur.execute("""
                CREATE TABLE IF NOT EXISTS agent_pair_performance (
                    id                 SERIAL PRIMARY KEY,
                    task_name          TEXT NOT NULL,
                    model              TEXT NOT NULL,
                    agent1             TEXT NOT NULL REFERENCES agents(title) ON DELETE CASCADE,
                    agent2             TEXT NOT NULL REFERENCES agents(title) ON DELETE CASCADE,
                    euclidean_distance FLOAT,
                    cosine_similarity  FLOAT,
                    response_length    FLOAT,
                    completion_time    FLOAT,
                    timestamp          TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    UNIQUE (task_name, model, agent1, agent2)
                );""")
            conn.commit()
    print("✅ Database setup completed.")

# =====================================
# Insert agents from CSV
# =====================================
def insert_agents_from_csv(csv_filepath: str):
    df = pd.read_csv(csv_filepath)
    df.columns = df.columns.str.strip()
    with psycopg2.connect(**DB_CONFIG, cursor_factory=LoggingCursor) as conn:
        with conn.cursor() as cur:
            for _, row in df.iterrows():
                try:
                    if pd.isna(row.get("Title")):
                        continue  # skip blank rows
                    title   = str(row["Title"]).strip()
                    if not title:
                        continue
                    profile = row["Profile"].strip() if pd.notna(row.get("Profile")) else None
                    memory  = row["Memory"].strip() if pd.notna(row.get("Memory")) else None
                    plan_txt = row["Planning"].strip() if pd.notna(row.get("Planning")) else "Unknown"
                    action  = row["Action"].strip() if pd.notna(row.get("Action")) else None
                    cap_acq = row["Capability Acquition"].strip() if pd.notna(row.get("Capability Acquition")) else None
                    soc_sci = row["Social science"].strip() if pd.notna(row.get("Social science")) else None
                    nat_sci = row["Natural Science"].strip() if pd.notna(row.get("Natural Science")) else None
                    eng     = row["Engineering"].strip() if pd.notna(row.get("Engineering")) else None
                    subj    = row["Subjective"].strip() if pd.notna(row.get("Subjective")) else None
                    obj     = row["Objective"].strip() if pd.notna(row.get("Objective")) else None
                    bench   = row["Benchmark"].strip() if pd.notna(row.get("Benchmark")) else None
                    pub     = row["Publication"].strip() if pd.notna(row.get("Publication")) else None
                    code_url= row["Code"].strip() if pd.notna(row.get("Code")) else None
                    paper_url=row["Paper"].strip() if pd.notna(row.get("Paper")) else None

                    cur.execute("SELECT id FROM planning_categories WHERE name = %s;", (plan_txt,))
                    cat = cur.fetchone()
                    cat_id = cat[0] if cat else None

                    sub_id = None
                    if "Subcategory" in df.columns and pd.notna(row.get("Subcategory")):
                        subcat = row["Subcategory"].strip()
                        cur.execute("SELECT id FROM planning_subcategories WHERE name = %s;", (subcat,))
                        res = cur.fetchone()
                        sub_id = res[0] if res else None

                    cur.execute("""
                        INSERT INTO agents
                          (title, profile, memory, planning_category_id, planning_subcategory_id,
                           action, capability_acquisition, social_science, natural_science,
                           engineering, subjective, objective, benchmark, publication, code_url, paper_url)
                        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                        ON CONFLICT (title) DO NOTHING;
                    """, (title,profile,memory,cat_id,sub_id,action,cap_acq,soc_sci,nat_sci,
                          eng,subj,obj,bench,pub,code_url,paper_url))
                    print(f"  ✅ {title}")
                except Exception as e:
                    print(f"  ⚠️ {row.get('Title', '??')}: {e}")
                    conn.rollback()
                else:
                    conn.commit()

# =====================================
# Insert tasks from CSV
# =====================================
def insert_tasks_from_csv(csv_filepath: str):
    df = pd.read_csv(csv_filepath)
    df.columns = df.columns.str.strip()
    with psycopg2.connect(**DB_CONFIG, cursor_factory=LoggingCursor) as conn:
        with conn.cursor() as cur:
            for _, row in df.iterrows():
                cur.execute("""
                    INSERT INTO tasks (task_name, query_text, query_type, domain, difficulty)
                    VALUES (%s, %s, %s, %s, %s)
                    ON CONFLICT (task_name) DO NOTHING;
                """, (
                    row["task_name"].strip(),
                    row["query_text"].strip(),
                    row.get("query_type", "default"),
                    row.get("domain"),
                    row.get("difficulty"),
                ))
        conn.commit()
    print(f"✅ Tasks inserted from {csv_filepath}.")

# =====================================
# Response storage (replaces generated_response array)
# ON CONFLICT DO NOTHING = free resume / idempotent
# =====================================
def store_response(agent_title: str, task_name: str, model: str, response: str):
    with psycopg2.connect(**DB_CONFIG, cursor_factory=LoggingCursor) as conn:
        with conn.cursor() as cur:
            cur.execute("""
                INSERT INTO agent_responses (agent_title, task_name, model, response)
                VALUES (%s, %s, %s, %s)
                ON CONFLICT (agent_title, task_name, model) DO NOTHING;
            """, (agent_title, task_name, model, response))
        conn.commit()

# =====================================
# Generic LLM call
# =====================================
def run_agent(agent_name: str, query_text: str, model: str, task_name: str,
              store: bool = True) -> str:
    print(f"  🤖 {agent_name} | {model} | {task_name}")
    prompt = f"System: {_ACTIVE_SYSTEM_PROMPT}\nUser: {query_text}\nAssistant:"
    try:
        r = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json={"model": model, "prompt": prompt, "stream": False},
            timeout=REQUEST_TIMEOUT,
        )
        if r.status_code == 200:
            answer = clean_response(r.json().get("response", "").strip())
            if len(answer) > MAX_RESPONSE_LEN:
                answer = answer[:MAX_RESPONSE_LEN] + "..."
            if answer and store:
                store_response(agent_name, task_name, model, answer)
            return answer or "No valid response received."
        print(f"  ❌ HTTP {r.status_code} ({agent_name})")
    except requests.exceptions.Timeout:
        print(f"  ⏱️ Timeout ({agent_name})")
    except Exception as e:
        print(f"  ❌ {agent_name}: {e}")
    return "No valid response received."

# =====================================
# Individual agent wrappers (all accept task_name)
# =====================================
def run_cot_langchain(query_text, model, task_name):
    return run_agent("Chain of Thought (CoT)", query_text, model, task_name)

def run_zero_shot_cot(query_text, model, task_name):
    return run_agent("Zero-Shot CoT", query_text + " Let's think step by step.", model, task_name)

def run_reprompting(query_text, model, task_name):
    base = "Please provide detailed, step-by-step reasoning, verifying each step."
    answer = run_agent("Re-Prompting", query_text + " " + base, model, task_name, store=False)
    if len(answer) < 50 or "step" not in answer.lower():
        refine = ("The previous response lacked step-by-step reasoning. "
                  "Re-generate by breaking down the problem and validating each step.")
        answer = run_agent("Re-Prompting", query_text + " " + refine, model, task_name, store=True)
    else:
        store_response("Re-Prompting", task_name, model, answer)
    return answer

def run_rewoo(query_text, model, task_name):
    candidate = run_agent("ReWOO", query_text, model, task_name, store=False)
    if not candidate or candidate == "No valid response received.":
        return candidate
    final = (candidate + " Observation: No anomalies detected in the environment.")[:MAX_RESPONSE_LEN]
    store_response("ReWOO", task_name, model, final)
    return final

def run_hugginggpt(query_text, model, task_name):
    instr = (" Decompose the task into sub-tasks, indicate the best HuggingFace model per sub-task,"
             " then integrate the outcomes into a concise answer.")
    return run_agent("HuggingGPT", query_text + instr, model, task_name)

def run_tree_of_thought(query_text, model, task_name):
    init_prompt = (f"System: {_ACTIVE_SYSTEM_PROMPT}\nUser: {query_text}\nAssistant: "
                   "Generate 3 distinct candidate reasoning steps, each on a new line.")
    try:
        r = requests.post(f"{OLLAMA_BASE_URL}/api/generate",
                          json={"model": model, "prompt": init_prompt, "stream": False},
                          timeout=REQUEST_TIMEOUT)
        candidates = ([clean_response(l) for l in r.json().get("response","").split("\n") if l.strip()]
                      if r.status_code == 200 else [])
    except Exception:
        candidates = []
    if not candidates:
        store_response("Tree-of-Thought (ToT)", task_name, model, "No valid chain of thought generated.")
        return "No valid chain of thought generated."

    def expand(c):
        prompt = (f"System: {_ACTIVE_SYSTEM_PROMPT}\nUser: Expand this reasoning step: '{c}' "
                  "with step-by-step detail.")
        try:
            r = requests.post(f"{OLLAMA_BASE_URL}/api/generate",
                              json={"model": model, "prompt": prompt, "stream": False},
                              timeout=REQUEST_TIMEOUT)
            return clean_response(r.json().get("response","")) if r.status_code == 200 else ""
        except Exception:
            return ""

    with ThreadPoolExecutor(max_workers=3) as ex:
        expansions = list(ex.map(expand, candidates))
    best = max([(c + " " + e)[:MAX_RESPONSE_LEN] for c, e in zip(candidates, expansions)], key=len)
    store_response("Tree-of-Thought (ToT)", task_name, model, best)
    return best

def run_llm_planner(query_text, model, task_name):
    examples = []
    if os.path.exists("knn_set.pkl"):
        try:
            import pickle as _pk
            with open("knn_set.pkl", "rb") as f:
                examples = _pk.load(f)
        except Exception:
            pass
    if not examples:
        examples = [
            "Example 1: Delivery route — identify stops; compute distances; apply TSP.",
            "Example 2: Cooking — list ingredients; sequence steps; factor cook times.",
        ]
    prompt = (f"System: {_ACTIVE_SYSTEM_PROMPT}\nFew-shot examples:\n"
              + "\n\n".join(examples)
              + f"\n\nUser: {query_text}\nAssistant:")
    try:
        r = requests.post(f"{OLLAMA_BASE_URL}/api/generate",
                          json={"model": model, "prompt": prompt, "stream": False},
                          timeout=REQUEST_TIMEOUT)
        if r.status_code == 200:
            answer = clean_response(r.json().get("response","").strip())[:MAX_RESPONSE_LEN]
            if answer:
                store_response("LLM-Planner", task_name, model, answer)
            return answer or "No valid response received."
    except Exception as e:
        print(f"  ❌ LLM-Planner: {e}")
    return "No valid response received."

def run_react(query_text, model, task_name):
    instr = (" Follow ReAct: 'Thought:' → 'Action:' → 'Observation:' → final answer. "
             f"Max {MAX_RESPONSE_LEN} characters.")
    return run_agent("ReAct", query_text + instr, model, task_name)

def run_llm_plus_p(query_text, model, task_name):
    instr = (" Translate to a PDDL problem, simulate a planner, convert the plan back "
             "to a concise natural-language answer.")
    return run_agent("LLM+P", query_text + instr, model, task_name)

def run_inner_monologue(query_text, model, task_name):
    return run_agent("Inner Monologue",
                     query_text + " Narrate your internal reasoning step by step, then give the answer.",
                     model, task_name)

def run_llm4rl(query_text, model, task_name):
    instr = (" Generate an initial plan; simulate an RL reward signal; "
             "refine iteratively to maximise reward; output the refined plan concisely.")
    return run_agent("LLM4RL", query_text + instr, model, task_name)

def run_chatcot(query_text, model, task_name):
    instr = (" Generate a chain-of-thought; reflect on each step; "
             "summarise and give a concise final answer.")
    return run_agent("ChatCoT", query_text + instr, model, task_name)

def run_tptu(query_text, model, task_name):
    instr = (" Draft an initial answer; simulate human advisor feedback; "
             "refine based on that feedback; output the refined answer concisely.")
    return run_agent("TPTU", query_text + instr, model, task_name)

def run_self_refine(query_text, model, task_name):
    initial = run_agent("Self-Refine", query_text, model, task_name, store=False)
    refine  = (query_text + f" Initial Answer: {initial} "
               "Review and refine for clarity, correctness, and conciseness.")
    return run_agent("Self-Refine", refine, model, task_name, store=True)

def run_selfcheck(query_text, model, task_name):
    initial = run_agent("SelfCheck", query_text, model, task_name, store=False)
    check   = (query_text + f" Initial Answer: {initial} "
               "Identify errors or inconsistencies, then produce a corrected final answer.")
    return run_agent("SelfCheck", check, model, task_name, store=True)

# =====================================
# Run all 15 agents on one (task, model) — agents run in parallel
# =====================================
AGENT_REGISTRY = {
    "Chain of Thought (CoT)": run_cot_langchain,
    "Zero-Shot CoT":          run_zero_shot_cot,
    "Re-Prompting":           run_reprompting,
    "ReWOO":                  run_rewoo,
    "HuggingGPT":             run_hugginggpt,
    "Tree-of-Thought (ToT)":  run_tree_of_thought,
    "LLM-Planner":            run_llm_planner,
    "ReAct":                  run_react,
    "LLM+P":                  run_llm_plus_p,
    "Inner Monologue":        run_inner_monologue,
    "LLM4RL":                 run_llm4rl,
    "ChatCoT":                run_chatcot,
    "TPTU":                   run_tptu,
    "Self-Refine":            run_self_refine,
    "SelfCheck":              run_selfcheck,
}

def process_task(task_name: str, query_text: str, query_type: str, model: str,
                 agents: Optional[dict] = None):
    """Run all (or selected) agents on one task with one model. Agents execute in parallel."""
    global _ACTIVE_SYSTEM_PROMPT
    _ACTIVE_SYSTEM_PROMPT = get_system_prompt(query_type)

    registry = agents or AGENT_REGISTRY
    responses = {}
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {
            ex.submit(fn, query_text, model, task_name): name
            for name, fn in registry.items()
        }
        # No timeout on as_completed — the per-request requests.post(timeout=REQUEST_TIMEOUT)
        # is the real cap. as_completed timeout fires for the whole batch which is too short
        # for multi-call agents (ToT: initial + 3 expand; Self-Refine: 2 passes).
        try:
            for future in as_completed(futures):
                name = futures[future]
                try:
                    responses[name] = future.result(timeout=REQUEST_TIMEOUT + 30 if REQUEST_TIMEOUT else None)
                except TimeoutError:
                    print(f"  ⏱️ {name}: timed out — skipping")
                    responses[name] = "No valid response received."
                except Exception as e:
                    print(f"  ❌ {name}: {e}")
                    responses[name] = "No valid response received."
        except TimeoutError:
            for future, name in futures.items():
                if name not in responses:
                    print(f"  ⏱️ {name}: batch timeout — skipping")
                    responses[name] = "No valid response received."
    print(f"  ✅ {task_name} | {model} — {len(responses)} agents done.")
    return responses

# =====================================
# Outer loop: all models × all tasks
# Models run sequentially; agents run in parallel within each (model, task)
# Already-completed (agent, task, model) triples are skipped automatically.
# =====================================
def process_all(tasks_csv: str, models: list = MODELS, agents: Optional[dict] = None):
    tasks_df = pd.read_csv(tasks_csv)
    tasks_df.columns = tasks_df.columns.str.strip()
    for model in models:
        print(f"\n🔹 Model: {model}")
        for _, row in tasks_df.iterrows():
            print(f"  📋 Task: {row['task_name']} [{row['query_type']}]")
            process_task(
                task_name  = row["task_name"],
                query_text = row["query_text"],
                query_type = row.get("query_type", "default"),
                model      = model,
                agents     = agents,
            )
    print("\n🚀 process_all complete.")

# =====================================
# Semantic search (unchanged, works across all stored responses)
# =====================================
def semantic_search_responses(query: str, top_k: int = 3,
                               model_filter: Optional[str] = None) -> List[str]:
    engine = create_engine(DB_URI)
    where  = f"WHERE model = '{model_filter}'" if model_filter else ""
    df = pd.read_sql(f"SELECT response FROM agent_responses {where}", engine)
    responses = df["response"].tolist()
    if not responses:
        print("No responses in database.")
        return []
    embeddings = embed_model.encode(responses, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    _, idxs = index.search(embed_model.encode([query], convert_to_numpy=True), top_k)
    return [responses[i] for i in idxs[0]]

# =====================================
# Entry point
# =====================================
RESET_DB    = False       # set True for a completely clean slate
TASKS_CSV   = "tasks.csv"
AGENTS_CSV  = "agents.csv"

def main():
    if RESET_DB:
        drop_existing_table()
    setup_database()
    populate_planning_taxonomy()
    insert_agents_from_csv(AGENTS_CSV)
    insert_tasks_from_csv(TASKS_CSV)
    print("✅ Setup complete.")
    process_all(TASKS_CSV, MODELS)

if __name__ == "__main__":
    main()


In [ ]:
# Block B: Embedding-Driven Task Performance and Stability Analysis
# Runs for ALL models in MODELS in a single execution — no switch needed.
import psycopg2
import pandas as pd
import numpy as np
import faiss
import time
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity

AGENT_TITLES = [
    "Chain of Thought (CoT)", "Zero-Shot CoT", "Re-Prompting",
    "ReWOO", "HuggingGPT", "Tree-of-Thought (ToT)",
    "LLM-Planner", "ReAct", "LLM+P",
    "Inner Monologue", "LLM4RL", "ChatCoT", "TPTU",
    "Self-Refine", "SelfCheck",
]
_ph = ",".join(["%s"] * len(AGENT_TITLES))

def run_block_b(model: str):
    print(f"\n{'='*55}\n  Block B — {model}\n{'='*55}")

    # ── 1. Fetch ──────────────────────────────────────────────
    with psycopg2.connect(**DB_CONFIG) as conn:
        with conn.cursor() as cur:
            cur.execute(
                f"""SELECT ar.agent_title AS title, ar.response, ar.task_name AS idx
                    FROM agent_responses ar
                    WHERE ar.model = %s AND ar.agent_title IN ({_ph})
                    ORDER BY ar.task_name;""",
                [model] + AGENT_TITLES,
            )
            df = pd.DataFrame(cur.fetchall(), columns=["title", "response", "idx"])
    print(f"  ✅ Fetched {len(df)} rows.")
    if df.empty:
        print(f"  ⚠️  No responses for {model} — skipping.")
        return

    # ── 2. Group by task ──────────────────────────────────────
    grouped = defaultdict(dict)
    for _, row in df.iterrows():
        grouped[row["idx"]][row["title"]] = row["response"]

    # ── 3. FAISS index per task ───────────────────────────────
    faiss_indexes = {}
    for task_id, resp_dict in grouped.items():
        resps = list(resp_dict.values())
        if len(resps) < 2:
            continue
        emb = embed_model.encode(resps, convert_to_numpy=True).astype(np.float32)
        idx = faiss.IndexFlatL2(emb.shape[1])
        idx.add(emb)
        faiss_indexes[task_id] = idx
        print(f"  ✅ '{task_id}': indexed {len(resps)} responses.")

    # ── 4. Stability check ────────────────────────────────────
    def tag(ed, cs):
        if   ed < 0.3 and cs > 0.9: return "✅ Stable"
        elif ed > 1.0 and cs < 0.5: return "❌ Diverging"
        else:                        return "🔄 Mixed"

    for task_id in grouped:
        print(f"\n  🔹 {task_id} | {model}")
        if task_id not in faiss_indexes:
            continue
        idx    = faiss_indexes[task_id]
        n      = idx.ntotal
        stored = np.zeros((n, idx.d), dtype=np.float32)
        for i in range(n): idx.reconstruct(i, stored[i])
        names = list(grouped[task_id].keys())
        vmap  = {names[i]: stored[i] for i in range(n)}
        for i, a in enumerate(names):
            for b in names[i+1:]:
                ed = float(np.linalg.norm(vmap[a] - vmap[b]))
                cs = float(cosine_similarity([vmap[a]], [vmap[b]])[0][0])
                print(f"    {a} ↔ {b}: ED={ed:.4f}  CosSim={cs:.4f}  {tag(ed, cs)}")

    # ── 5. Save task_performance (1 aggregate row per task per model) ──
    # "5 rows" per model = 1 row × 5 tasks; total 10 rows when both models are done.
    sql_tp = """
        INSERT INTO task_performance
          (task_name, model, avg_euclidean_distance, avg_cosine_similarity,
           response_length, completion_time)
        VALUES (%s,%s,%s,%s,%s,%s)
        ON CONFLICT (task_name, model) DO UPDATE SET
          avg_euclidean_distance = EXCLUDED.avg_euclidean_distance,
          avg_cosine_similarity  = EXCLUDED.avg_cosine_similarity,
          response_length        = EXCLUDED.response_length,
          completion_time        = EXCLUDED.completion_time;
    """
    with psycopg2.connect(**DB_CONFIG, cursor_factory=LoggingCursor) as conn:
        with conn.cursor() as cur:
            for task_id in grouped:
                if task_id not in faiss_indexes:
                    continue
                idx    = faiss_indexes[task_id]
                n      = idx.ntotal
                start  = time.time()
                stored = np.zeros((n, idx.d), dtype=np.float32)
                for i in range(n): idx.reconstruct(i, stored[i])
                resps  = list(grouped[task_id].values())
                dists, sims = [], []
                for i in range(n):
                    for j in range(i+1, n):
                        dists.append(float(np.linalg.norm(stored[i] - stored[j])))
                        sims.append(float(cosine_similarity([stored[i]], [stored[j]])[0][0]))
                cur.execute(sql_tp, (
                    str(task_id), model,
                    float(np.mean(dists)) if dists else 0.0,
                    float(np.mean(sims))  if sims  else 0.0,
                    int(np.mean([len(r) for r in resps])),
                    float(time.time() - start),
                ))
                print(f"  ✅ task_performance saved: {task_id} | {model}")
        conn.commit()

    # ── 6. Cleaned CSV ────────────────────────────────────────
    with psycopg2.connect(**DB_CONFIG) as conn:
        df_raw = pd.read_sql("""
            SELECT ar.agent_title AS title, ar.response, ar.task_name AS idx, ar.model,
                   a.profile, a.memory, a.action, a.capability_acquisition,
                   a.social_science, a.natural_science, a.engineering, a.benchmark
            FROM agent_responses ar
            JOIN agents a ON a.title = ar.agent_title
            WHERE ar.model = %(model)s
            ORDER BY ar.task_name;
        """, conn, params={"model": model})
    df_clean = df_raw[df_raw["response"].notna() & (df_raw["response"].str.strip() != "")]
    out = f"cleaned_agent_responses_{model.replace(':', '_').replace('.', '_')}.csv"
    df_clean.to_csv(RESULTS_DIR / out, index=False)
    print(f"  ✅ Cleaned CSV → {out}  ({len(df_clean)}/{len(df_raw)} rows)")


# ── Run for every model in one shot ──────────────────────────
for _m in MODELS:
    run_block_b(_m)

print("\n✅ Block B complete for all models.")


In [ ]:
# Block C: PCA-Driven Task Matrix Construction and Pairwise Performance Evaluation
# Runs for ALL models in MODELS in a single execution — no switch needed.
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from collections import defaultdict
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

def run_block_c(model: str):
    print(f"\n{'='*55}\n  Block C — {model}\n{'='*55}")

    # ── 1. Fetch ──────────────────────────────────────────────
    with psycopg2.connect(**DB_CONFIG) as conn:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT ar.agent_title AS title, ar.response, ar.task_name AS idx
                FROM agent_responses ar
                WHERE ar.model = %s
                ORDER BY ar.task_name;
            """, (model,))
            df = pd.DataFrame(cur.fetchall(), columns=["title", "response", "idx"])
    print(f"  ✅ Fetched {len(df)} rows.")
    if df.empty:
        print(f"  ⚠️  No responses for {model} — skipping.")
        return

    # ── 2. Group by task ──────────────────────────────────────
    grouped = defaultdict(dict)
    for _, row in df.iterrows():
        grouped[row["idx"]][row["title"]] = row["response"]

    # ── 3. Global PCA ─────────────────────────────────────────
    all_emb    = embed_model.encode(df["response"].tolist(), convert_to_numpy=True)
    all_scaled = MinMaxScaler().fit_transform(all_emb)
    pca        = PCA(n_components=2)
    pca.fit(all_scaled)
    pca_scaled = MinMaxScaler().fit_transform(pca.transform(all_scaled))
    print(f"  PCA variance: {pca.explained_variance_ratio_.round(4)} "
          f"(total={pca.explained_variance_ratio_.sum():.4f})")

    # ── 4. Task matrix ────────────────────────────────────────
    task_matrix = defaultdict(dict)
    for task_id in df["idx"].unique():
        for row_idx in df[df["idx"] == task_id].index:
            title  = df.at[row_idx, "title"]
            vector = pca_scaled[df.index.get_loc(row_idx)]
            if not np.all(vector == 0):
                task_matrix[task_id][title] = {
                    "Total Score":    float(np.sum(np.abs(vector))),
                    "Reduced Vector": vector,
                }
    print(f"  Task matrix built for {len(task_matrix)} tasks.")

    safe = model.replace(":", "_").replace(".", "_")
    rows_out = [{"Task": t, "Agent": a, "Model": model, "Total Score": d["Total Score"]}
                for t in task_matrix for a, d in task_matrix[t].items()]
    pd.DataFrame(rows_out).to_csv(RESULTS_DIR / f"normalized_task_matrix_{safe}.csv", index=False)
    print(f"  ✅ Task matrix CSV saved.")

    # ── 5. Heatmaps ───────────────────────────────────────────
    valid_labels, valid_vectors = [], []
    for task_id in sorted(task_matrix):
        for agent in sorted(task_matrix[task_id]):
            v = task_matrix[task_id][agent]["Reduced Vector"]
            if np.any(v):
                valid_labels.append(f"{agent} (T:{task_id})")
                valid_vectors.append(v)

    if len(valid_labels) >= 2:
        V = np.array(valid_vectors)
        n = len(valid_labels)
        tick_step = 1 if n < 40 else 2  # seaborn integer = show every nth label (no misalignment)

        fig, ax = plt.subplots(figsize=(min(15, n/2), min(12, n/2)))
        sns.heatmap(cosine_similarity(V), annot=False, cmap="coolwarm",
                    xticklabels=tick_step, yticklabels=tick_step, ax=ax)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
        ax.set_title(f"Cosine Similarity — {model}")
        fig.tight_layout()
        fig.savefig(RESULTS_DIR / f"cosine_heatmap_{safe}.png", dpi=150)
        plt.close(fig)

        euc = np.sqrt(((V[:, None] - V) ** 2).sum(axis=2))
        fig, ax = plt.subplots(figsize=(min(15, n/2), min(12, n/2)))
        sns.heatmap(euc, annot=n < 20, fmt=".2f", cmap="viridis",
                    xticklabels=tick_step, yticklabels=tick_step, ax=ax)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
        ax.set_title(f"Euclidean Distance — {model}")
        fig.tight_layout()
        fig.savefig(RESULTS_DIR / f"euclidean_heatmap_{safe}.png", dpi=150)
        plt.close(fig)

    # ── 6. Pairwise performance → agent_pair_performance ──────
    sql_pair = """
        INSERT INTO agent_pair_performance
          (task_name, model, agent1, agent2,
           euclidean_distance, cosine_similarity, response_length, completion_time)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
        ON CONFLICT (task_name, model, agent1, agent2) DO UPDATE SET
          euclidean_distance = EXCLUDED.euclidean_distance,
          cosine_similarity  = EXCLUDED.cosine_similarity,
          response_length    = EXCLUDED.response_length,
          completion_time    = EXCLUDED.completion_time;
    """
    with psycopg2.connect(**DB_CONFIG, cursor_factory=LoggingCursor) as conn:
        with conn.cursor() as cur:
            for task_id in sorted(grouped):
                resp_dict = grouped[task_id]
                agents    = list(resp_dict)
                for i, a1 in enumerate(agents):
                    for a2 in agents[i+1:]:
                        if a1 not in task_matrix[task_id] or a2 not in task_matrix[task_id]:
                            print(f"  ❌ Missing vector: {a1} or {a2} in {task_id}")
                            continue
                        v1, v2 = (task_matrix[task_id][a1]["Reduced Vector"],
                                  task_matrix[task_id][a2]["Reduced Vector"])
                        r1, r2 = resp_dict[a1], resp_dict[a2]
                        start  = time.time()
                        cur.execute(sql_pair, (
                            str(task_id), model, a1, a2,
                            float(np.linalg.norm(v1 - v2)),
                            float(cosine_similarity([v1], [v2])[0][0]),
                            float((len(r1) + len(r2)) / 2),
                            float(time.time() - start),
                        ))
        conn.commit()

    with psycopg2.connect(**DB_CONFIG) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT COUNT(*) FROM agent_pair_performance WHERE model = %s;", (model,))
            count = cur.fetchone()[0]
    print(f"  ✅ agent_pair_performance: {count} total rows for {model}")


# ── Run for every model in one shot ──────────────────────────
for _m in MODELS:
    run_block_c(_m)

print("\n✅ Block C complete for all models.")


In [ ]:
# Block D: Heatmaps for a single task/model
# Switch ACTIVE_MODEL or ACTIVE_TASK to compare across models/tasks.
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ACTIVE_MODEL = MODELS[0]    # e.g. "tinyllama"  or MODELS[1] = "qwen3.5:27b"
ACTIVE_TASK  = None         # None = auto-pick first available task

with psycopg2.connect(**DB_CONFIG) as conn:
    df_pair = pd.read_sql(
        "SELECT * FROM agent_pair_performance WHERE model = %s ORDER BY task_name, agent1, agent2;",
        conn, params=(ACTIVE_MODEL,)
    )

print(f"Rows for {ACTIVE_MODEL}: {len(df_pair)}")
print(df_pair.head())

# Auto-pick first task if not specified
if ACTIVE_TASK is None:
    ACTIVE_TASK = df_pair["task_name"].iloc[0] if not df_pair.empty else None

df_task = df_pair[df_pair["task_name"] == ACTIVE_TASK] if ACTIVE_TASK else pd.DataFrame()

if df_task.empty:
    print(f"No data for task={ACTIVE_TASK}, model={ACTIVE_MODEL}.")
else:
    euc_mat = df_task.pivot(index="agent1", columns="agent2", values="euclidean_distance")
    cos_mat = df_task.pivot(index="agent1", columns="agent2", values="cosine_similarity")

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(euc_mat, annot=True, fmt=".3f", cmap="viridis", ax=ax)
    ax.set_title(f"Euclidean Distance — Task: {ACTIVE_TASK} | Model: {ACTIVE_MODEL}")
    ax.set_xlabel("Agent 2"); ax.set_ylabel("Agent 1")
    plt.xticks(rotation=45); fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"euclidean_{ACTIVE_TASK}_{ACTIVE_MODEL.replace(':','_').replace('.','_')}.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cos_mat, annot=True, fmt=".3f", cmap="coolwarm", ax=ax)
    ax.set_title(f"Cosine Similarity — Task: {ACTIVE_TASK} | Model: {ACTIVE_MODEL}")
    ax.set_xlabel("Agent 2"); ax.set_ylabel("Agent 1")
    plt.xticks(rotation=45); fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"cosine_{ACTIVE_TASK}_{ACTIVE_MODEL.replace(':','_').replace('.','_')}.png", dpi=150)
    plt.close(fig)


In [ ]:
# Block E: Agent-Level Aggregated Metrics across tasks (Euclidean & Cosine)
# Switch ACTIVE_MODEL to compare models.
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt

ACTIVE_MODEL = MODELS[0]

with psycopg2.connect(**DB_CONFIG) as conn:
    df_pair = pd.read_sql(
        "SELECT * FROM agent_pair_performance WHERE model = %s ORDER BY task_name, agent1, agent2;",
        conn, params=(ACTIVE_MODEL,)
    )

print(f"Rows for {ACTIVE_MODEL}: {len(df_pair)}")
print(df_pair.head())

def agent_level_avg(df, metric):
    """Duplicate rows so each agent appears on both sides of the pair."""
    df1 = df[["task_name", "agent1", metric]].rename(columns={"agent1": "agent"})
    df2 = df[["task_name", "agent2", metric]].rename(columns={"agent2": "agent"})
    combined = pd.concat([df1, df2], ignore_index=True)
    return combined.groupby(["task_name", "agent"], as_index=False)[metric].mean()

# --- Euclidean distance ---
df_ed = agent_level_avg(df_pair, "euclidean_distance")
pivot_ed = df_ed.pivot(index="task_name", columns="agent", values="euclidean_distance")

fig, ax = plt.subplots(figsize=(12, 8))
for agent in pivot_ed.columns:
    vals = pivot_ed[agent].dropna()
    ax.scatter(vals.index, vals.values, label=agent, s=80, alpha=0.7)
ax.set_xlabel("Task"); ax.set_ylabel("Avg Euclidean Distance")
ax.set_title(f"Avg Euclidean Distance per Agent — {ACTIVE_MODEL}")
ax.legend(title="Agent", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=45); fig.tight_layout()
fig.savefig(RESULTS_DIR / f"avg_ed_{ACTIVE_MODEL.replace(':','_').replace('.','_')}.png", dpi=150)
plt.close(fig)

# --- Cosine similarity ---
df_cos = agent_level_avg(df_pair, "cosine_similarity")
pivot_cos = df_cos.pivot(index="task_name", columns="agent", values="cosine_similarity")

fig, ax = plt.subplots(figsize=(12, 8))
for agent in pivot_cos.columns:
    vals = pivot_cos[agent].dropna()
    ax.scatter(vals.index, vals.values, label=agent, s=80, alpha=0.7)
ax.set_xlabel("Task"); ax.set_ylabel("Avg Cosine Similarity")
ax.set_title(f"Avg Cosine Similarity per Agent — {ACTIVE_MODEL}")
ax.legend(title="Agent", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=45); fig.tight_layout()
fig.savefig(RESULTS_DIR / f"avg_cosine_{ACTIVE_MODEL.replace(':','_').replace('.','_')}.png", dpi=150)
plt.close(fig)


In [ ]:
# Block F: KMeans Clustering — single task, single model
# Switch ACTIVE_MODEL or ACTIVE_TASK to compare.
import psycopg2
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

ACTIVE_MODEL = MODELS[0]
ACTIVE_TASK  = None   # None = auto-pick first available task

with psycopg2.connect(**DB_CONFIG) as conn:
    df_all = pd.read_sql(
        "SELECT * FROM agent_pair_performance WHERE model = %s ORDER BY task_name;",
        conn, params=(ACTIVE_MODEL,)
    )

if ACTIVE_TASK is None:
    ACTIVE_TASK = df_all["task_name"].iloc[0] if not df_all.empty else None

df_task = df_all[df_all["task_name"] == ACTIVE_TASK]
print(f"Task: {ACTIVE_TASK} | Model: {ACTIVE_MODEL} | Rows: {len(df_task)}")
print(df_task.head())

features = ["euclidean_distance", "cosine_similarity", "response_length", "completion_time"]
X_scaled = StandardScaler().fit_transform(df_task[features].values)

k = 3
df_task = df_task.copy()
df_task["cluster"] = KMeans(n_clusters=k, random_state=42, n_init="auto").fit_predict(X_scaled)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA explained variance: {pca.explained_variance_ratio_.round(4)} "
      f"(total={pca.explained_variance_ratio_.sum():.4f})")
df_task["pca1"] = X_pca[:, 0]
df_task["pca2"] = X_pca[:, 1]

fig, ax = plt.subplots(figsize=(10, 8))
sns.scatterplot(data=df_task, x="pca1", y="pca2", hue="cluster", palette="viridis", s=100, ax=ax)
for _, row in df_task.iterrows():
    ax.text(row["pca1"]+0.02, row["pca2"]+0.02, f"{row['agent1']} vs {row['agent2']}", fontsize=7)
ax.set_title(f"KMeans (k=3) — Task: {ACTIVE_TASK} | Model: {ACTIVE_MODEL}")
fig.tight_layout()
fig.savefig(RESULTS_DIR / f"kmeans_{ACTIVE_TASK}_{ACTIVE_MODEL.replace(':','_').replace('.','_')}.png", dpi=150)
plt.close(fig)


In [ ]:
# Block G: KMeans with Planning Taxonomy — all tasks, single model
# Switch ACTIVE_MODEL to compare.
import psycopg2
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

ACTIVE_MODEL = MODELS[0]

PLANNING_AGENTS = {
    "Planning Without Feedback": {
        "Single-Path Reasoning": ["Chain of Thought (CoT)", "Zero-Shot CoT", "Re-Prompting"],
        "Multi-Path Reasoning":  ["ReWOO", "HuggingGPT", "Tree-of-Thought (ToT)"],
        "External Planner":      ["LLM-Planner", "ReAct", "LLM+P"],
    },
    "Planning With Feedback": {
        "Environment Feedback": ["Inner Monologue", "LLM4RL"],
        "Human Feedback":       ["ChatCoT", "TPTU"],
        "Model Feedback":       ["Self-Refine", "SelfCheck"],
    },
}
agent_to_subcat = {a: sub for grp in PLANNING_AGENTS.values()
                   for sub, agents in grp.items() for a in agents}

with psycopg2.connect(**DB_CONFIG) as conn:
    df_pair = pd.read_sql(
        "SELECT * FROM agent_pair_performance WHERE model = %s ORDER BY task_name, agent1, agent2;",
        conn, params=(ACTIVE_MODEL,)
    )

df_pair["agent1_subcat"] = df_pair["agent1"].map(agent_to_subcat)
df_pair["agent2_subcat"] = df_pair["agent2"].map(agent_to_subcat)
df_pair["pair_category"] = df_pair.apply(
    lambda r: "-".join(sorted([str(r["agent1_subcat"]), str(r["agent2_subcat"])])), axis=1)

features = ["euclidean_distance", "cosine_similarity", "response_length", "completion_time"]
X_scaled = StandardScaler().fit_transform(df_pair[features].values)
df_pair = df_pair.copy()
df_pair["cluster"] = KMeans(n_clusters=5, random_state=42, n_init="auto").fit_predict(X_scaled)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA explained variance: {pca.explained_variance_ratio_.round(4)} "
      f"(total={pca.explained_variance_ratio_.sum():.4f})")
df_pair["pca1"] = X_pca[:, 0]
df_pair["pca2"] = X_pca[:, 1]

fig, ax = plt.subplots(figsize=(14, 10))
sns.scatterplot(data=df_pair, x="pca1", y="pca2", hue="pair_category",
                style="cluster", palette="viridis", s=100, ax=ax)
for _, row in df_pair.iterrows():
    ax.text(row["pca1"]+0.02, row["pca2"]+0.02,
            f"{row['agent1']} vs {row['agent2']}", fontsize=7)
ax.set_title(f"KMeans (k=5) Taxonomy Clustering — {ACTIVE_MODEL}")
ax.legend(title="Pair Category", bbox_to_anchor=(1.05, 1), loc="upper left")
fig.tight_layout()
fig.savefig(RESULTS_DIR / f"kmeans_taxonomy_{ACTIVE_MODEL.replace(':','_').replace('.','_')}.png", dpi=150)
plt.close(fig)
print(df_pair[["task_name","agent1","agent2","pair_category","cluster"]].head(20))


In [ ]:
# Block H: Complexity-Adjusted Agent Ranking — single task, single model
# combined metric = cosine_similarity − euclidean_distance (higher = better)
# Switch ACTIVE_MODEL or ACTIVE_TASK to compare.
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ACTIVE_MODEL = MODELS[0]
ACTIVE_TASK  = None   # None = auto-pick first task

COMPLEXITY = {
    "Inner Monologue": 1, "LLM4RL": 1,
    "ChatCoT": 2, "TPTU": 2,
    "Self-Refine": 3, "SelfCheck": 3,
    "Chain of Thought (CoT)": 4, "Zero-Shot CoT": 4, "Re-Prompting": 4,
    "ReWOO": 5, "HuggingGPT": 5, "Tree-of-Thought (ToT)": 5,
    "LLM-Planner": 6, "ReAct": 6, "LLM+P": 6,
}

with psycopg2.connect(**DB_CONFIG) as conn:
    df_all = pd.read_sql(
        "SELECT * FROM agent_pair_performance WHERE model = %s ORDER BY task_name;",
        conn, params=(ACTIVE_MODEL,)
    )

if ACTIVE_TASK is None:
    ACTIVE_TASK = df_all["task_name"].iloc[0] if not df_all.empty else None

df_pair = df_all[df_all["task_name"] == ACTIVE_TASK]
print(f"Task: {ACTIVE_TASK} | Model: {ACTIVE_MODEL} | Rows: {len(df_pair)}")

agents = set(df_pair["agent1"]) | set(df_pair["agent2"])
records = []
for agent in agents:
    rows = df_pair[(df_pair["agent1"] == agent) | (df_pair["agent2"] == agent)]
    if rows.empty:
        continue
    avg_ed  = rows["euclidean_distance"].mean()
    avg_cos = rows["cosine_similarity"].mean()
    records.append({
        "agent":           agent,
        "avg_ed":          avg_ed,
        "avg_cosine":      avg_cos,
        "avg_resp_len":    rows["response_length"].mean(),
        "avg_time":        rows["completion_time"].mean(),
        "combined_metric": float(avg_cos) - float(avg_ed),  # higher = better
        "complexity":      COMPLEXITY.get(agent, 10),
    })

df_agents = (pd.DataFrame(records)
             .sort_values(["complexity", "combined_metric"], ascending=[True, False])
             .reset_index(drop=True))

print("\nRanked agents:")
print(df_agents.to_string(index=False))
best = df_agents.iloc[0]
print(f"\n🏆 Best: {best['agent']} (complexity={best['complexity']}, combined={best['combined_metric']:.4f})")

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df_agents["complexity"], df_agents["combined_metric"], s=100, c="skyblue", edgecolors="k")
for _, row in df_agents.iterrows():
    ax.text(row["complexity"]+0.06, row["combined_metric"]+0.005, row["agent"], fontsize=9)
ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
ax.set_xlabel("Complexity (lower = simpler)")
ax.set_ylabel("Combined Metric (cosine − ED)")
ax.set_title(f"Complexity vs Performance — Task: {ACTIVE_TASK} | Model: {ACTIVE_MODEL}")
fig.tight_layout()
fig.savefig(RESULTS_DIR / f"complexity_{ACTIVE_TASK}_{ACTIVE_MODEL.replace(':','_').replace('.','_')}.png", dpi=150)
plt.close(fig)


In [ ]:
# Block I: Cross-Model Comparison
# Requires: Blocks A, B, C all run for BOTH models.
# Three views:
#   1. Side-by-side bars — task-level metrics (tinyllama vs qwen per task)
#   2. Cross-model agreement heatmap — for each (agent, task): how similar are the two models' answers?
#   3. Combined metric scatter — all agents, both models overlaid; complexity on x-axis
#   4. Winner table — which model wins per task
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

M1, M2 = MODELS[0], MODELS[1]   # "tinyllama", "qwen3.5:27b"

COMPLEXITY = {
    "Inner Monologue": 1, "LLM4RL": 1,
    "ChatCoT": 2, "TPTU": 2,
    "Self-Refine": 3, "SelfCheck": 3,
    "Chain of Thought (CoT)": 4, "Zero-Shot CoT": 4, "Re-Prompting": 4,
    "ReWOO": 5, "HuggingGPT": 5, "Tree-of-Thought (ToT)": 5,
    "LLM-Planner": 6, "ReAct": 6, "LLM+P": 6,
}

# ── 1. Task-level metrics side-by-side ───────────────────────
# task_performance: 1 aggregate row per (task, model)
# = avg of all 15 agents' pairwise cosine/ED for that task
with psycopg2.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT * FROM task_performance WHERE model = ANY(%s) ORDER BY task_name, model;",
            ([M1, M2],)
        )
        cols     = [d[0] for d in cur.description]
        df_perf  = pd.DataFrame(cur.fetchall(), columns=cols)

print(f"✅ task_performance rows: {len(df_perf)}")
print(df_perf[["task_name","model","avg_cosine_similarity","avg_euclidean_distance","response_length"]]
      .to_string(index=False))

for metric, ylabel, title in [
    ("avg_cosine_similarity",  "Avg Cosine Similarity\n(higher = agents more uniform)",   "Inter-Agent Cosine Similarity"),
    ("avg_euclidean_distance", "Avg Euclidean Distance\n(higher = agents more diverse)",   "Inter-Agent Euclidean Distance"),
    ("response_length",        "Avg Response Length (chars)",                              "Response Length"),
]:
    pivot = df_perf.pivot(index="task_name", columns="model", values=metric)
    ax    = pivot.plot(kind="bar", figsize=(10, 5), width=0.6,
                       color=["steelblue", "coral"], edgecolor="k", linewidth=0.5)
    ax.set_title(f"{title} — {M1} vs {M2}")
    ax.set_xlabel("Task"); ax.set_ylabel(ylabel)
    ax.legend(title="Model")
    plt.xticks(rotation=30, ha="right"); plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"compare_{metric}.png", dpi=150)
    plt.close()

# ── 2. Cross-model response agreement ────────────────────────
# For each (agent, task): embed both models' responses and compute cosine similarity.
# High = models give similar answers; Low = models diverge on that agent+task.
with psycopg2.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT agent_title, task_name, model, response FROM agent_responses WHERE model = ANY(%s);",
            ([M1, M2],)
        )
        df_resp = pd.DataFrame(cur.fetchall(), columns=["agent", "task", "model", "response"])

s1 = df_resp[df_resp["model"] == M1].set_index(["agent", "task"])["response"]
s2 = df_resp[df_resp["model"] == M2].set_index(["agent", "task"])["response"]
common = s1.index.intersection(s2.index)

if len(common) == 0:
    print(f"⚠️  No overlapping (agent, task) pairs between {M1} and {M2}. "
          "Run Block A for both models first.")
else:
    # Batch encode — 2 encode calls total (fast)
    texts_m1 = [s1.loc[k] for k in common]
    texts_m2 = [s2.loc[k] for k in common]
    emb1 = embed_model.encode(texts_m1, convert_to_numpy=True)
    emb2 = embed_model.encode(texts_m2, convert_to_numpy=True)
    # Element-wise cosine similarity
    norms = np.linalg.norm(emb1, axis=1) * np.linalg.norm(emb2, axis=1) + 1e-8
    cross_cos = (emb1 * emb2).sum(axis=1) / norms

    df_cross = pd.DataFrame(
        [{"agent": a, "task": t, "cross_cosine": float(cs)}
         for (a, t), cs in zip(common, cross_cos)]
    )
    print(f"\n✅ Cross-model similarity: {len(df_cross)} (agent, task) pairs.")
    print("Most divergent pairs (models disagree most):")
    print(df_cross.sort_values("cross_cosine").head(10).to_string(index=False))

    cross_pivot = df_cross.pivot(index="agent", columns="task", values="cross_cosine")
    fig, ax = plt.subplots(figsize=(10, 9))
    sns.heatmap(cross_pivot, annot=True, fmt=".2f", cmap="RdYlGn",
                vmin=0, vmax=1, linewidths=0.4, ax=ax)
    ax.set_title(
        f"Cross-Model Agreement: {M1} vs {M2}\n"
        "1.0 = identical answers  |  0.0 = completely different"
    )
    ax.set_xlabel("Task"); ax.set_ylabel("Agent")
    plt.xticks(rotation=30, ha="right"); plt.tight_layout()
    plt.savefig(RESULTS_DIR / "cross_model_agreement.png", dpi=150)
    plt.close()

# ── 3. Combined metric scatter — both models overlaid ─────────
with psycopg2.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT * FROM agent_pair_performance WHERE model = ANY(%s);",
            ([M1, M2],)
        )
        cols     = [d[0] for d in cur.description]
        df_pairs = pd.DataFrame(cur.fetchall(), columns=cols)

records = []
for model in [M1, M2]:
    df_m = df_pairs[df_pairs["model"] == model]
    for agent in set(df_m["agent1"]) | set(df_m["agent2"]):
        rows_a = df_m[(df_m["agent1"] == agent) | (df_m["agent2"] == agent)]
        if rows_a.empty:
            continue
        records.append({
            "agent":      agent,
            "model":      model,
            "combined":   float(rows_a["cosine_similarity"].mean())
                          - float(rows_a["euclidean_distance"].mean()),
            "complexity": COMPLEXITY.get(agent, 10),
        })

df_rank = pd.DataFrame(records)
colors   = {M1: "steelblue", M2: "coral"}

fig, ax = plt.subplots(figsize=(12, 6))
for model in [M1, M2]:
    sub = df_rank[df_rank["model"] == model]
    ax.scatter(sub["complexity"], sub["combined"],
               label=model, color=colors[model],
               s=110, alpha=0.85, edgecolors="k", linewidths=0.5)
    for _, r in sub.iterrows():
        ax.annotate(r["agent"],
                    xy=(r["complexity"], r["combined"]),
                    xytext=(4, 3), textcoords="offset points", fontsize=7)
ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
ax.set_xlabel("Complexity (1=simple, 6=complex)")
ax.set_ylabel("Combined Metric (cosine − ED, higher = better)")
ax.set_title(f"Agent Performance Across All Tasks — {M1} vs {M2}")
ax.legend(title="Model")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "compare_combined_metric.png", dpi=150)
plt.close(fig)

# ── 4. Winner table ───────────────────────────────────────────
winner_rows = []
for task in df_perf["task_name"].unique():
    sub = df_perf[df_perf["task_name"] == task].set_index("model")
    if M1 not in sub.index or M2 not in sub.index:
        continue
    c1 = sub.loc[M1, "avg_cosine_similarity"] - sub.loc[M1, "avg_euclidean_distance"]
    c2 = sub.loc[M2, "avg_cosine_similarity"] - sub.loc[M2, "avg_euclidean_distance"]
    winner_rows.append({
        "task":            task,
        f"{M1}_score":     round(c1, 4),
        f"{M2}_score":     round(c2, 4),
        "winner":          M1 if c1 > c2 else (M2 if c2 > c1 else "Tie"),
        "margin":          round(abs(c1 - c2), 4),
    })

df_winners = pd.DataFrame(winner_rows).sort_values("margin", ascending=False)
print("\n🏆 Model Comparison — Winner per Task (sorted by margin):")
print(df_winners.to_string(index=False))
m1_wins = (df_winners["winner"] == M1).sum()
m2_wins = (df_winners["winner"] == M2).sum()
print(f"\n  {M1}: {m1_wins} task wins")
print(f"  {M2}: {m2_wins} task wins")
